Adding Data from the CWTS dataset (IPP + SNIP) to Our Datasets by Year. + Imputing NAN vallues (about 10% of All data with ML)

# CWTS Journal indicators (adding IPP and SNIP Column to our data)

In [14]:
import pandas as pd
df_CWTS=pd.read_excel("CWTS Journal Indicators March 2024.xlsx",sheet_name="Journals")

## 0. check how much data is overlapping

In [10]:
data=df["Source title"].unique()

# 1. add from CWTS metrics to Each One Depending on the Year:

In [180]:
#how many of those deck all mine:
import os
clean=pd.read_csv(os.path.join("Prepared Datasets", "CLEAN_Prepd_scimagojr 2023.csv"))

In [8]:
found=0
for value in clean["Title"]:
    for new in data:
        if value==new:
            found+=1
            break
print(found)
print(clean.shape[0])

10855
12193


In [46]:
clean=pd.read_csv(os.path.join("Prepared Datasets", "CLEAN_Prepd_scimagojr 2023.csv"))

In [49]:
df_CWTS_year=df_CWTS[df_CWTS["Year"]==2023].drop_duplicates(subset=["Source title"],ignore_index=True)
df_CWTS_year.rename(columns={"Source title": "Title"},inplace=True)

In [45]:
df_CWTS_year=df_CWTS[df_CWTS["Year"]==year].drop_duplicates(subset=["Source title"],ignore_index=True)
df_CWTS_year

,Source title,Source type,Print ISSN,Electronic ISSN,ASJC field IDs,Year,Citing source,P,IPP,IPP (lower bound),IPP (upper bound),SNIP,SNIP (lower bound),SNIP (upper bound),% self cit
0,2D Materials,Journal,-,2053-1583,1600; 2210; 2211; 2500; 3104,2023,1,578,4.868512,4.361592,5.493080,1.036967,0.913966,1.194119,0.026297
1,3 Biotech,Journal,2190-572X,2190-5738,1101; 1305; 2301,2023,1,1402,3.122682,2.917261,3.344508,0.751399,0.691957,0.812316,0.020786
2,3D Printing and Additive Manufacturing,Journal,2329-7662,2329-7670,2209; 2501,2023,1,118,4.076271,3.279661,4.991525,0.982129,0.758793,1.230072,0.060291
3,"3L: Language, Linguistics, Literature",Journal,0128-5157,2550-2247,1203; 1208; 3310,2023,1,169,0.994083,0.751479,1.260355,0.818289,0.597436,1.093936,0.232143
4,452°F,Journal,-,2013-3294,1208,2023,0,84,0.023810,0.000000,0.059524,0.163889,0.000000,0.457143,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27856,Zutot,Journal,1571-7283,1875-0214,1200; 1212,2023,1,39,0.051282,0.000000,0.128205,0.435013,0.000000,1.124668,0.500000
27857,ZWF Zeitschrift für Wirtschaftlichen Fabrikbet...,Journal,0947-0085,2511-0896,1408; 1803; 2200,2023,1,573,0.479930,0.394415,0.568935,0.463056,0.373729,0.559420,0.232727
27858,Zygon,Journal,0591-2385,1467-9744,1212; 3304; 3316,2023,1,174,0.419540,0.327586,0.522989,0.746745,0.503957,1.047897,0.287671
27859,Zygote,Journal,0967-1994,1469-8730,1307; 1309,2023,1,199,1.582915,1.341709,1.859296,0.474680,0.392330,0.565725,0.060317


In [182]:
attr=['Title', 'SJR', 'H index', 'IPP', 'SNIP','Total Docs. (2023)', 'Total Docs. (3years)',
       'Total Refs.', 'Est. value (USD) (2023)', 'Total Cites (3years)',
       'Self-Cites/Total Cites (3years)', 'Uncited Docs./Total Docs. (3years)',
       'Citable Docs. (3years)', 'Cites / Doc. (2years)', 'Ref. / Doc.',
       'Coverage_Duration', 'Publisher' ]
resulting=clean.merge(df_CWTS_year[["Title", "IPP", "SNIP"]],how='left', on="Title")

# 2. Clean the Data: get the data from ML learning to impute in the last two columns!

In [199]:
def ImputeValuesIntoDataFrame(resulting):
    good_data=resulting[resulting.notna().SNIP]
    from sklearn.model_selection import train_test_split
    X=good_data.drop(columns=["IPP", "SNIP","Title","Publisher"])
    X=X.to_numpy()
    y=good_data[["IPP", "SNIP"]]
    y=y.to_numpy()
    X_train0, X_train1,y_train0,y_train1 =train_test_split(
        X, y, test_size=0.2, random_state=42)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train0=scaler.fit_transform(X_train0)
    X_train1=scaler.fit_transform(X_train1)
    from sklearn.ensemble import RandomForestRegressor
    
    RFR = RandomForestRegressor(max_depth=6, random_state=0)
    
    RFR.fit(X_train0, y_train0)
    
    print ('Accuracy scores for filling NAN with Linear Regression model are:')
    NANLR_MAE , NANLR_MSE , NANLR_R2_sc = acc_chk(y_train1 , RFR.predict(X_train1))
    
    print('LR Model Train Score is : ' , RFR.score(X_train0, y_train0))
    print('LR Model Test Score is : ' , RFR.score(X_train1, y_train1))
    
    #prepare Test Data:
    missing=resulting[resulting.SNIP.isna()]
    X_test=missing.drop(columns=["IPP", "SNIP","Title","Publisher"])
    X_test=X_test.to_numpy()
    X_test=scaler.fit_transform(X_test)
    
    RESULT=RFR.predict(X_test)
    indeces=resulting[resulting.SNIP.isna()].index
    Result=pd.DataFrame(RESULT,columns=["IPP","SNIP"], index=indeces)

    resulting.fillna(Result, inplace=True)
    return resulting

### Test Data

In [188]:
missing=resulting[resulting.SNIP.isna()]
y_test= missing.loc[:,["IPP", "SNIP"]]
X_test=missing.drop(columns=["IPP", "SNIP","Title","Publisher"])

In [189]:
X_test

,SJR,H index,Total Docs. (2023),Total Docs. (3years),Total Refs.,Est. value (USD) (2023),Total Cites (3years),Self-Cites/Total Cites (3years),Uncited Docs./Total Docs. (3years),Citable Docs. (3years),Cites / Doc. (2years),Ref. / Doc.,Coverage_Duration
13,18.587,30,11,16,0,91445.0,754,0.000000,0.000000,16,44.86,0.00,5
48,12.179,432,388,1454,5208,12727543.0,18298,0.007815,0.407840,633,11.62,13.42,26
49,12.113,936,1390,4043,22032,37953443.0,66124,0.006700,0.407618,1369,14.46,15.85,203
67,10.247,189,194,575,4605,4875438.0,7095,0.011276,0.330435,222,12.56,23.74,13
79,9.434,106,164,504,4181,4132216.0,7592,0.007376,0.212302,218,12.20,25.49,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27008,0.100,5,35,90,0,357095.0,3,0.000000,0.966667,68,0.03,0.00,22
27056,0.100,4,21,77,1555,165635.0,1,0.000000,0.987013,77,0.02,74.05,16
27071,0.100,3,87,230,0,853544.0,1,0.000000,0.995652,224,0.01,0.00,21
27072,0.100,5,33,102,2055,420546.0,7,0.142857,0.941176,90,0.05,62.27,14


### Training data

In [190]:
good_data=resulting[resulting.notna().SNIP]

In [130]:
len(y_train1)

5202

In [131]:
from sklearn.model_selection import train_test_split
X=good_data.drop(columns=["IPP", "SNIP","Title","Publisher"])
X=X.to_numpy()
y=good_data[["IPP", "SNIP"]]
y=y.to_numpy()
X_train0, X_train1,y_train0,y_train1 =train_test_split(
    X, y, test_size=0.2, random_state=42)

In [132]:
from sklearn.preprocessing import StandardScaler
#y_train0=y_train0.to_numpy()
#y_train1=y_train1.to_numpy()
#X_train0=X_train0.to_numpy()
#X_train1=X_train1.to_numpy()
#y_test=y_test.to_numpy()
#X_test=X_test.to_numpy()
scaler = StandardScaler()
X_train0=scaler.fit_transform(X_train0)
X_train1=scaler.fit_transform(X_train1)

In [198]:
def acc_chk (y_tst , y_hat):
    # Mean absolute error
    MAE = metrics.mean_absolute_error(y_tst ,y_hat)
    
    # Mean Suared Error (MSE)
    MSE = metrics.mean_squared_error(y_tst ,y_hat)
    
    # R2-score
    R2_sc = metrics.r2_score(y_tst ,y_hat)
    
    print("Mean absolute error: %.2f" % MAE)
    print("Mean Squared Error : %.2f" % MSE)
    print("R2-score           : %.2f" % R2_sc )
    
    return MAE , MSE , R2_sc

In [134]:
from sklearn.ensemble import RandomForestRegressor

RFR = RandomForestRegressor(max_depth=6, random_state=0)

RFR.fit(X_train0, y_train0)

print ('Accuracy scores for filling NAN with Linear Regression model are:')
NANLR_MAE , NANLR_MSE , NANLR_R2_sc = acc_chk(y_train1 , RFR.predict(X_train1))

print('LR Model Train Score is : ' , RFR.score(X_train0, y_train0))
print('LR Model Test Score is : ' , RFR.score(X_train1, y_train1))

Accuracy scores for filling NAN with Linear Regression model are:
Mean absolute error: 0.41
Mean Squared Error : 0.94
R2-score           : 0.69
LR Model Train Score is :  0.8887737277641468
LR Model Test Score is :  0.6938160506935056


- Error is good enough
  

In [ ]:
#prepare Test Data:
X_test=X_test.to_numpy()
y_test=y_test.to_numpy()
X_test=scaler.fit_transform(X_test)

In [136]:
RESULT=RFR.predict(X_test)

In [191]:
indeces=resulting[resulting.SNIP.isna()].index
Result=pd.DataFrame(RESULT,columns=["IPP","SNIP"], index=indeces)

In [192]:
resulting.fillna(Result, inplace=True)

,IPP,SNIP
13,66.416836,13.355885
48,51.932259,11.071718
49,52.065266,11.019566
67,44.873573,9.815450
79,45.234025,9.953928
...,...,...
27008,2.353654,1.006780
27056,2.350826,1.008913
27071,2.270465,1.029039
27072,2.368669,1.006340


In [197]:
resulting.isna().sum()

Title                                 0
SJR                                   0
H index                               0
Total Docs. (2023)                    0
Total Docs. (3years)                  0
Total Refs.                           0
Est. value (USD) (2023)               0
Total Cites (3years)                  0
Self-Cites/Total Cites (3years)       0
Uncited Docs./Total Docs. (3years)    0
Citable Docs. (3years)                0
Cites / Doc. (2years)                 0
Ref. / Doc.                           0
Coverage_Duration                     0
Publisher                             0
IPP                                   0
SNIP                                  0
dtype: int64

# 3. Running on (1999-2023) Datasets [ no SNIP, IPP data for 2024]

In [207]:
# from IPP
import numpy as np
import time
start=time.time()

#loop through each one i have:
for year in range(1999,2023): #iteration over ALL databases years
    attr=['Title', 'SJR', 'H index', 'IPP', 'SNIP','Total Docs. ('+str(year)+')', 'Total Docs. (3years)',
       'Total Refs.', 'Est. value (USD) ('+str(year)+')', 'Total Cites (3years)',
       'Self-Cites/Total Cites (3years)', 'Uncited Docs./Total Docs. (3years)',
       'Citable Docs. (3years)', 'Cites / Doc. (2years)', 'Ref. / Doc.',
       'Coverage_Duration', 'Publisher' ]
    #how many of those deck all mine:
    import os
    clean=pd.read_csv(os.path.join("Prepared Datasets", "CLEAN_Prepd_scimagojr "+str(year)+".csv"))
    df_CWTS_year=df_CWTS[df_CWTS["Year"]==year].drop_duplicates(subset=["Source title"],ignore_index=True)
    #rename to merge:
    df_CWTS_year.rename(columns={"Source title": "Title"},inplace=True)
    #merge+rename:
    resulting=clean.merge(df_CWTS_year[["Title", "IPP", "SNIP"]],how='left', on="Title")[attr]
    #cleaning (Imputing Vaues with ML):
    resulting=ImputeValuesIntoDataFrame(resulting)
    
    resulting.to_csv(os.path.join("CWTS+Datasets", "TOTAL_scimagojr "+str(year)+".csv"), index=False)


end=time.time()

Accuracy scores for filling NAN with Linear Regression model are:
Mean absolute error: 0.26
Mean Squared Error : 0.43
R2-score           : 0.69
LR Model Train Score is :  0.8456183124653031
LR Model Test Score is :  0.6893763337431353
Accuracy scores for filling NAN with Linear Regression model are:
Mean absolute error: 0.25
Mean Squared Error : 0.33
R2-score           : 0.75
LR Model Train Score is :  0.837948152221636
LR Model Test Score is :  0.7536388749240843
Accuracy scores for filling NAN with Linear Regression model are:
Mean absolute error: 0.24
Mean Squared Error : 0.32
R2-score           : 0.76
LR Model Train Score is :  0.8520875308986173
LR Model Test Score is :  0.7592409561012395
Accuracy scores for filling NAN with Linear Regression model are:
Mean absolute error: 0.25
Mean Squared Error : 0.30
R2-score           : 0.77
LR Model Train Score is :  0.8582754952336662
LR Model Test Score is :  0.7719109762941312
Accuracy scores for filling NAN with Linear Regression model 

--some models taken from https://www.kaggle.com/code/alaatemimy/filling-of-missing-values-using-machine-learning